## KV Cache

During autoregressive generation, the model repeatedly processes one new token at a time. KV caching avoids recomputing the Key and Value tensors for tokens that have already been processed.

- **Prefill:** Process the full prompt and store the K/V tensors from each Transformer block.
- **Decode:** Process only the new token and reuse the cached K/V tensors.
- **Validation:** Compare the cached prediction with normal full-sequence computation.

The cached and full-sequence logits should match up to small floating-point differences.

In [3]:
import torch
import sys
sys.path.append("..")
from model.gpt import GPT
from config import (
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    INTERMEDIATE_SIZE,
    NUM_BLOCKS,
)

torch.manual_seed(42)

model = GPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    intermediate_size=INTERMEDIATE_SIZE,
    num_blocks=NUM_BLOCKS,
)

model.eval()

prompt_ids = torch.randint(0, VOCAB_SIZE, (1, 4))
next_token_id = torch.randint(0, VOCAB_SIZE, (1, 1))

with torch.no_grad():

    # Prefill
    prompt_logits, past_key_values = model(
        prompt_ids,
        use_cache=True,
    )

    # Decode one new token using cache
    cached_logits, _ = model(
        next_token_id,
        past_key_values=past_key_values,
        use_cache=True,
    )

    # Normal full-sequence computation
    full_ids = torch.cat([prompt_ids, next_token_id], dim=1)
    full_logits = model(full_ids)

cached_last = cached_logits[:, -1, :]
full_last = full_logits[:, -1, :]

print("Cached logits shape:", cached_logits.shape)
print("Full logits shape:  ", full_logits.shape)

print(
    "Cache matches full:",
    torch.allclose(cached_last, full_last, atol=1e-6),
)

assert torch.allclose(cached_last, full_last, atol=1e-6)

print("KV cache test passed.")

Cached logits shape: torch.Size([1, 1, 8192])
Full logits shape:   torch.Size([1, 5, 8192])
Cache matches full: True
KV cache test passed.
